# Real-World Data Wrangling: Macroeconomic Trends and Stock Market Valuations

This notebook implements an end-to-end data wrangling workflow to explore how U.S. macroeconomic growth and inflation relate to S&P 500 valuation metrics, long-term interest rates, and a gold futures/proxy series.

The workflow gathers raw data, assesses quality issues, cleans and validates the data, merges annual observations, stores outputs, and produces exploratory visualizations.

## Important interpretation notes

- The World Bank data is fetched directly for `country/USA`, not `country/all`, to avoid aggregate region/income-group rows.
- `GC=F` from Yahoo Finance is treated as a gold futures/proxy series, not physical spot gold.
- `sp500_annual_avg_yoy_change` and `gold_annual_avg_yoy_change` are year-over-year changes in annual average price levels, not true calendar-year total returns.
- `cape_yield` is computed as `100 / PE10`, so it is a CAPE-yield proxy.
- `cape_yield_minus_10y_yield` is a rough CAPE-yield minus 10-year Treasury-yield spread, not a full expected equity risk premium model.
- The merged annual sample covers 2000–2023, so the analysis is exploratory rather than causal.


In [ ]:
# Uncomment and run once in a fresh environment.
#!pip install -r requirements.txt


## 1. Imports and project constants

In [ ]:
from __future__ import annotations

import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import yfinance as yf

DATA_DIR = Path("data")
IMAGE_DIR = Path("images")
START_YEAR = 2000
END_YEAR = 2023

DATA_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 2. Helper functions

In [ ]:
def first_matching_column(
    columns: pd.Index,
    *,
    exact_names: set[str],
    prefixes: tuple[str, ...] = (),
) -> object | None:
    """Return the first column whose normalized name matches common variants."""
    for col in columns:
        col_name = str(col).strip()
        lowered = col_name.lower()
        if lowered in exact_names or any(lowered.startswith(prefix) for prefix in prefixes):
            return col
    return None


def fetch_world_bank_indicator(indicator: str, label: str) -> list:
    """Fetch one World Bank indicator directly for the United States."""
    url = (
        f"https://api.worldbank.org/v2/country/USA/indicator/{indicator}"
        f"?date={START_YEAR}:{END_YEAR}&format=json&per_page=2000"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if len(payload) < 2 or payload[1] is None:
        raise ValueError(f"No data returned for {label} ({indicator})")
    return payload


def normalize_world_bank_indicator(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Normalize nested World Bank records into a compact annual table."""
    out = pd.DataFrame(
        {
            "country_name": df["country"].apply(
                lambda value: value.get("value") if isinstance(value, dict) else np.nan
            ),
            "countryiso3code": df["countryiso3code"].replace("", np.nan),
            "date": pd.to_numeric(df["date"], errors="coerce").astype("Int64"),
            value_col: pd.to_numeric(df["value"], errors="coerce"),
        }
    )
    out["countryiso3code"] = out["countryiso3code"].astype("string").str.strip()
    return out


## 3. Gather and clean S&P 500/Shiller data

The S&P 500/Shiller dataset is downloaded from the public GitHub-hosted CSV, then filtered to the 2000–2023 analysis window.


In [ ]:
url_sp500 = "https://raw.githubusercontent.com/datasets/s-and-p-500/master/data/data.csv"
raw_sp_path = DATA_DIR / "sp500_shiller_raw.csv"

response_sp = requests.get(url_sp500, timeout=30)
response_sp.raise_for_status()
raw_sp_path.write_bytes(response_sp.content)

df_sp500_raw = pd.read_csv(raw_sp_path)
print("Loaded S&P 500/Shiller raw data:", df_sp500_raw.shape)
df_sp500_raw.head()


In [ ]:
df_sp500_clean = df_sp500_raw.copy()
df_sp500_clean["Date"] = pd.to_datetime(df_sp500_clean["Date"], errors="coerce")

numeric_cols = [
    "SP500",
    "Dividend",
    "Earnings",
    "Consumer Price Index",
    "Long Interest Rate",
    "Real Price",
    "Real Dividend",
    "Real Earnings",
    "PE10",
]
for col in numeric_cols:
    df_sp500_clean[col] = pd.to_numeric(df_sp500_clean[col], errors="coerce")

# PE10 appears as 0.0 in early rows where CAPE is not available.
df_sp500_clean["PE10"] = df_sp500_clean["PE10"].replace(0, np.nan)

df_sp500_clean = df_sp500_clean[
    df_sp500_clean["Date"].dt.year.between(START_YEAR, END_YEAR)
].copy()

required_sp500_cols = [
    "Date",
    "SP500",
    "Dividend",
    "Earnings",
    "Consumer Price Index",
    "Long Interest Rate",
    "PE10",
]
df_sp500_clean = df_sp500_clean.dropna(subset=required_sp500_cols)

assert df_sp500_clean["Date"].is_unique, "S&P 500 monthly dates should be unique"
assert df_sp500_clean["Date"].dt.year.min() == START_YEAR
assert df_sp500_clean["Date"].dt.year.max() == END_YEAR

df_sp500_clean.to_csv(DATA_DIR / "sp500_shiller_clean.csv", index=False)
print("Cleaned S&P 500/Shiller data:", df_sp500_clean.shape)
df_sp500_clean.head()


## 4. Gather and clean World Bank U.S. GDP growth and inflation data

The World Bank API is queried directly for the United States. This avoids aggregate rows such as income groups and regions that appear when using `country/all`.


In [ ]:
gdp_data = fetch_world_bank_indicator("NY.GDP.MKTP.KD.ZG", "GDP growth")
inflation_data = fetch_world_bank_indicator("FP.CPI.TOTL.ZG", "CPI inflation")

(DATA_DIR / "world_bank_gdp_raw.json").write_text(json.dumps(gdp_data, indent=2))
(DATA_DIR / "world_bank_inflation_raw.json").write_text(json.dumps(inflation_data, indent=2))

df_gdp_raw = pd.DataFrame(gdp_data[1])
df_inf_raw = pd.DataFrame(inflation_data[1])

print("GDP raw data:", df_gdp_raw.shape)
print("Inflation raw data:", df_inf_raw.shape)
display(df_gdp_raw.head())
display(df_inf_raw.head())


In [ ]:
df_gdp_clean = normalize_world_bank_indicator(df_gdp_raw, "gdp_growth")
df_inf_clean = normalize_world_bank_indicator(df_inf_raw, "inflation_rate")

df_world_bank_clean = pd.merge(
    df_gdp_clean,
    df_inf_clean,
    on=["country_name", "countryiso3code", "date"],
    how="inner",
    validate="one_to_one",
)

df_world_bank_clean = (
    df_world_bank_clean[df_world_bank_clean["countryiso3code"] == "USA"]
    .sort_values("date")
    .reset_index(drop=True)
)

assert df_world_bank_clean["countryiso3code"].eq("USA").all()
assert df_world_bank_clean["date"].is_unique
assert df_world_bank_clean["date"].min() == START_YEAR
assert df_world_bank_clean["date"].max() == END_YEAR
assert df_world_bank_clean[["gdp_growth", "inflation_rate"]].notna().all().all()

df_world_bank_clean.to_csv(DATA_DIR / "world_bank_clean.csv", index=False)
print("Cleaned World Bank data:", df_world_bank_clean.shape)
df_world_bank_clean.head()


## 5. Gather and clean Yahoo Finance gold futures/proxy data

`GC=F` is a futures-based Yahoo Finance series. This notebook uses it as a practical proxy for gold exposure, not as a physical spot-gold benchmark.


In [ ]:
df_gold_raw = yf.download(
    "GC=F",
    start=f"{START_YEAR}-01-01",
    end=f"{END_YEAR + 1}-01-01",
    progress=False,
    auto_adjust=False,
)

if df_gold_raw.empty:
    raise ValueError("No gold data returned from yfinance for GC=F")

print("Gold futures/proxy raw data:", df_gold_raw.shape)
df_gold_raw.head()


In [ ]:
df_gold_clean = df_gold_raw.copy()

if isinstance(df_gold_clean.columns, pd.MultiIndex):
    df_gold_clean.columns = [
        "_".join(str(part) for part in col if part).strip("_")
        for col in df_gold_clean.columns.to_flat_index()
    ]

df_gold_clean = df_gold_clean.reset_index()
df_gold_clean.to_csv(DATA_DIR / "gold_prices_raw.csv", index=False)

date_col = first_matching_column(
    df_gold_clean.columns,
    exact_names={"date", "datetime", "index"},
    prefixes=("date_", "datetime_"),
)
close_col = first_matching_column(
    df_gold_clean.columns,
    exact_names={"close"},
    prefixes=("close_",),
)
volume_col = first_matching_column(
    df_gold_clean.columns,
    exact_names={"volume"},
    prefixes=("volume_",),
)

missing = [
    name
    for name, col in {"date": date_col, "close": close_col, "volume": volume_col}.items()
    if col is None
]
if missing:
    raise ValueError(
        f"Could not find required gold data column(s): {', '.join(missing)}. "
        f"Columns returned by yfinance: {[str(col) for col in df_gold_clean.columns]}"
    )

df_gold_clean = df_gold_clean.rename(
    columns={
        date_col: "Date",
        close_col: "Close",
        volume_col: "Volume",
    }
)
df_gold_clean["Date"] = pd.to_datetime(df_gold_clean["Date"], errors="coerce")
df_gold_clean["Close"] = pd.to_numeric(df_gold_clean["Close"], errors="coerce")
df_gold_clean["Volume"] = pd.to_numeric(df_gold_clean["Volume"], errors="coerce")
df_gold_clean = df_gold_clean.dropna(subset=["Date", "Close"])
df_gold_clean = df_gold_clean[
    df_gold_clean["Date"].dt.year.between(START_YEAR, END_YEAR)
].copy()

df_gold_clean.to_csv(DATA_DIR / "gold_prices_clean.csv", index=False)
print("Cleaned gold futures/proxy data:", df_gold_clean.shape)
df_gold_clean.head()


## 6. Aggregate to annual frequency and merge with one-to-one validation

In [ ]:
df_sp500_annual = (
    df_sp500_clean.assign(Year=df_sp500_clean["Date"].dt.year)
    .groupby("Year", as_index=False)[
        [
            "SP500",
            "Dividend",
            "Earnings",
            "Consumer Price Index",
            "Long Interest Rate",
            "PE10",
        ]
    ]
    .mean()
)

df_gold_annual = (
    df_gold_clean.assign(Year=df_gold_clean["Date"].dt.year)
    .groupby("Year", as_index=False)
    .agg(gold_close=("Close", "mean"), gold_volume=("Volume", "mean"))
)

df_wb_annual = df_world_bank_clean.rename(columns={"date": "Year"})[
    ["Year", "gdp_growth", "inflation_rate"]
].copy()
df_wb_annual["Year"] = df_wb_annual["Year"].astype(int)

for name, df in {
    "S&P annual": df_sp500_annual,
    "World Bank annual": df_wb_annual,
    "Gold annual": df_gold_annual,
}.items():
    assert df["Year"].is_unique, f"{name} has duplicate years"

df_merged = pd.merge(
    df_sp500_annual,
    df_wb_annual,
    on="Year",
    how="inner",
    validate="one_to_one",
)

df_merged = pd.merge(
    df_merged,
    df_gold_annual,
    on="Year",
    how="inner",
    validate="one_to_one",
)

df_merged = df_merged.sort_values("Year").reset_index(drop=True)

assert df_merged["Year"].is_unique
assert df_merged["Year"].min() == START_YEAR
assert df_merged["Year"].max() == END_YEAR
assert df_merged[["SP500", "PE10", "gdp_growth", "inflation_rate", "gold_close"]].notna().all().all()

print("Merged annual data:", df_merged.shape)
df_merged.head()


## 7. Feature engineering with explicit column names

The annual price-change fields are year-over-year changes in annual average levels. They are not total calendar-year investment returns.


In [ ]:
df_merged["cape_yield"] = 100 / df_merged["PE10"]

df_merged["cape_yield_minus_10y_yield"] = (
    df_merged["cape_yield"] - df_merged["Long Interest Rate"]
)

df_merged["sp500_annual_avg_yoy_change"] = (
    df_merged["SP500"].pct_change() * 100
)

df_merged["gold_annual_avg_yoy_change"] = (
    df_merged["gold_close"].pct_change() * 100
)

df_merged["gdp_growth_lag1"] = df_merged["gdp_growth"].shift(1)

retired_columns = {
    "earnings_yield",
    "equity_risk_premium",
    "sp500_annual_return",
    "gold_annual_return",
}
assert not retired_columns.intersection(df_merged.columns)

df_merged.to_csv(DATA_DIR / "macro_stock_merged.csv", index=False)
df_merged.head()


## 8. Store cleaned data in SQLite

In [ ]:
db_path = DATA_DIR / "macro_stock_data.db"

with sqlite3.connect(db_path) as conn:
    df_sp500_clean.to_sql("sp500_shiller_clean", conn, if_exists="replace", index=False)
    df_world_bank_clean.to_sql("world_bank_clean", conn, if_exists="replace", index=False)
    df_gold_clean.to_sql("gold_prices_clean", conn, if_exists="replace", index=False)
    df_merged.to_sql("macro_stock_merged", conn, if_exists="replace", index=False)

print(f"Stored cleaned tables in {db_path}")


## 9. Exploratory visualizations

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df_merged, x="gdp_growth", y="PE10")
plt.title("Shiller PE10 vs. Annual U.S. GDP Growth")
plt.xlabel("GDP growth (%)")
plt.ylabel("Shiller PE10 / CAPE")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "visual1_pe10_vs_gdp.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(
    data=df_merged,
    x="inflation_rate",
    y="Long Interest Rate",
    label="10-year Treasury yield",
)
sns.regplot(
    data=df_merged,
    x="inflation_rate",
    y="cape_yield",
    label="CAPE yield proxy",
)
plt.title("Inflation vs. Yield Measures")
plt.xlabel("CPI inflation (%)")
plt.ylabel("Yield / proxy (%)")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR / "visual2_inflation_vs_yields.png", dpi=150)
plt.show()


In [ ]:
corr_cols = [
    "gdp_growth",
    "inflation_rate",
    "SP500",
    "PE10",
    "Long Interest Rate",
    "cape_yield",
    "cape_yield_minus_10y_yield",
    "gold_close",
    "sp500_annual_avg_yoy_change",
    "gold_annual_avg_yoy_change",
]

plt.figure(figsize=(10, 7))
sns.heatmap(df_merged[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Exploratory)")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "visual3_correlation_heatmap.png", dpi=150)
plt.show()


In [ ]:
df_regimes = df_merged.copy()
df_regimes["inflation_regime"] = pd.cut(
    df_regimes["inflation_rate"],
    bins=[-np.inf, 2, 4, np.inf],
    labels=["Low (<2%)", "Moderate (2-4%)", "High (>4%)"],
)

regime_summary = (
    df_regimes.groupby("inflation_regime", observed=True)[
        ["sp500_annual_avg_yoy_change", "gold_annual_avg_yoy_change"]
    ]
    .agg(["count", "mean", "median", "std"])
)

plot_data = (
    df_regimes.groupby("inflation_regime", observed=True)[
        ["sp500_annual_avg_yoy_change", "gold_annual_avg_yoy_change"]
    ]
    .mean()
    .reset_index()
    .melt(
        id_vars="inflation_regime",
        var_name="Series",
        value_name="Average YoY change",
    )
)

plt.figure(figsize=(8, 5))
sns.barplot(data=plot_data, x="inflation_regime", y="Average YoY change", hue="Series")
plt.title("Average Annual-Mean Price Changes by Inflation Regime")
plt.xlabel("Inflation regime")
plt.ylabel("Average YoY change in annual average price (%)")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "visual4_inflation_regimes.png", dpi=150)
plt.show()

regime_summary


In [ ]:
lag_data = df_merged.dropna(
    subset=["sp500_annual_avg_yoy_change", "gdp_growth"]
).copy()

plt.figure(figsize=(8, 5))
sns.regplot(data=lag_data, x="sp500_annual_avg_yoy_change", y="gdp_growth")
plt.title("S&P 500 Annual-Average YoY Change vs. Same-Year GDP Growth")
plt.xlabel("S&P 500 annual-average YoY change (%)")
plt.ylabel("GDP growth (%)")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "visual5_lagged_lead_gdp.png", dpi=150)
plt.show()


## 10. PE10 outlier check

The outlier threshold is computed directly from the current data instead of being hard-coded in the narrative.


In [ ]:
q1 = df_merged["PE10"].quantile(0.25)
q3 = df_merged["PE10"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

pe10_outliers = df_merged[df_merged["PE10"] > upper_bound][["Year", "PE10"]]

print(f"PE10 upper outlier bound: {upper_bound:.2f}")
pe10_outliers


## 11. Interpretation

This analysis is exploratory. The small annual sample can suggest patterns, but it does not establish causality. The annual-average price-change columns are not total investment returns. Stronger conclusions would require formal lag tests, Granger-causality tests, out-of-sample forecasting, true total-return equity data, and/or a benchmark spot-gold series.


## 12. Validate committed outputs

In [ ]:
from scripts.validate_outputs import main as validate_outputs

validate_outputs()
